# Module 4: Attack Module

This notebook is the final split-out stage of Module 4 and the only split notebook that runs federated learning. It loads the centralized MobileNetV3 target checkpoint and MobileNetV2 surrogate checkpoint, evaluates surrogate and transfer attacks, then runs clean-vs-attacked FL for the algorithm selected in `attack_module_config.yaml`. It can also sweep multiple FL poisoning recipes or compare several algorithms when the corresponding optional run modes are enabled.

Run this notebook from the `4_Adversarial_FL/` directory after `train_v3.ipynb` and `train_surrogate.ipynb`.


## Stage Goal

Use the saved target and surrogate artifacts to separate these questions:

1. How vulnerable is the surrogate to random noise, FGSM, and PGD?
2. Do surrogate-crafted examples transfer to the centralized MobileNetV3 target?
3. Starting from the same V3 checkpoint, how does the selected FL algorithm behave with honest clients versus malicious clients?
4. When enabled, how do random noise, FGSM, and PGD poisoning recipes compare in FL?
5. When enabled, how do multiple FL algorithms compare under the same poisoning recipe?

The attack-recipe sweep, malicious-fraction sweep, and algorithm comparison remain guarded by `attack_module.run_attack_recipe_sweep`, `attack_module.run_malicious_fraction_sweep`, and `attack_module.run_algorithm_comparison` in `attack_module_config.yaml`.


## 1. Notebook Setup

Load the FL algorithm registry, attack functions, model classes, and evaluation helpers.


In [ ]:
from copy import deepcopy
from pathlib import Path
import os

import json
import yaml
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from algos import (
    SUPPORTED_ALGORITHMS,
    canonical_algorithm_name,
    get_algorithm_server_class,
    select_malicious_client_ids,
)
from attacks import ATTACK_FUNCTIONS, AttackFn, denormalize_pixels, pixel_linf_norm
from malicious_client import MaliciousClient
from model import MobileNetV2Transfer, MobileNetV3Transfer
from util_functions import (
    create_data,
    evaluate_fn,
    select_validation_subset,
    set_logger,
    set_seed,
    target_label_prediction_rate,
)


## 2. Configuration and Artifact Setup

Load `attack_module_config.yaml`, resolve the device, validate the selected algorithm, optional comparison algorithms, and attack settings, and create the artifact directory.


In [ ]:
CONFIG_PATH = Path("attack_module_config.yaml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError("Could not locate attack_module_config.yaml in the working directory")
with CONFIG_PATH.open() as f:
    CONFIG = yaml.safe_load(f)


def _merge_dicts(base: dict | None, override: dict | None) -> dict:
    merged = deepcopy(base or {})
    for key, value in (override or {}).items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = _merge_dicts(merged[key], value)
        else:
            merged[key] = deepcopy(value)
    return merged


def _resolve_surrogate_training_config(config: dict) -> tuple[str, dict]:
    selection = deepcopy(config.get("surrogate_training", {}))
    legacy = deepcopy(config.get("surrogate", {}))
    profiles = deepcopy(config.get("surrogate_training_profiles", {}))
    default_profile = "quick"
    profile = str(selection.get("profile", legacy.get("profile", default_profile)))
    if profile not in profiles:
        available = ", ".join(sorted(profiles)) or "none"
        raise ValueError(
            f"Unknown surrogate_training.profile={profile!r}. Available profiles: {available}."
        )

    cfg = _merge_dicts(legacy, profiles.get(profile, {}))
    cfg = _merge_dicts(cfg, selection)
    optimizer_cfg = cfg.get("optimizer", {}) if isinstance(cfg.get("optimizer", {}), dict) else {}
    criterion_cfg = cfg.get("criterion", {}) if isinstance(cfg.get("criterion", {}), dict) else {}

    if "lr" in optimizer_cfg:
        cfg["learning_rate"] = optimizer_cfg["lr"]
        cfg["lr"] = optimizer_cfg["lr"]
    if "weight_decay" in optimizer_cfg:
        cfg["weight_decay"] = optimizer_cfg["weight_decay"]
    if "label_smoothing" in criterion_cfg:
        cfg["label_smoothing"] = criterion_cfg["label_smoothing"]
    return profile, cfg


global_config = deepcopy(CONFIG.get("global_config", {}))
data_config = deepcopy(CONFIG.get("data_config", {}))
model_config = deepcopy(CONFIG.get("model_config", {}))
alg_configs = deepcopy(CONFIG.get("algorithms", {}))
attack_defaults = deepcopy(CONFIG.get("attack", {}))
artifact_config = deepcopy(CONFIG.get("artifacts", {}))
SURROGATE_PROFILE, surrogate_training_config = _resolve_surrogate_training_config(CONFIG)
attack_module_config = deepcopy(CONFIG.get("attack_module", {}))


def _canonical_or_raw_algorithm(name: str) -> str:
    try:
        return canonical_algorithm_name(name)
    except KeyError:
        return str(name)


def _algorithm_config(algorithm_name: str) -> dict:
    canonical = canonical_algorithm_name(algorithm_name)
    for configured_name, configured in alg_configs.items():
        try:
            if canonical_algorithm_name(configured_name) == canonical:
                return deepcopy(configured or {})
        except KeyError:
            continue
    return {}


def _canonical_algorithm_sequence(values) -> list[str]:
    if isinstance(values, str):
        values = [values]
    names = []
    for value in values or []:
        canonical = _canonical_or_raw_algorithm(value)
        if canonical not in names:
            names.append(canonical)
    return names


RAW_SELECTED_ALGORITHM = attack_module_config.get(
    "selected_algorithm",
    attack_module_config.get("algorithm", "FedAvg"),
)
SELECTED_ALGORITHM = canonical_algorithm_name(RAW_SELECTED_ALGORITHM)
AVAILABLE_ALGORITHMS = sorted({_canonical_or_raw_algorithm(name) for name in alg_configs})
SELECTED_ALG_CONFIG = _algorithm_config(SELECTED_ALGORITHM)
DEFAULT_FED_CONFIG = deepcopy(SELECTED_ALG_CONFIG.get("fed_config", {}))
DEFAULT_OPTIM_CONFIG = deepcopy(SELECTED_ALG_CONFIG.get("optim_config", {}))

RUN_SURROGATE_ATTACKS = bool(attack_module_config.get("run_surrogate_attacks", True))
RUN_TRANSFER_ATTACKS = bool(attack_module_config.get("run_transfer_attacks", True))
RUN_FEDERATED_POISONING = bool(attack_module_config.get("run_federated_poisoning", True))
RUN_ATTACK_RECIPE_SWEEP = bool(attack_module_config.get("run_attack_recipe_sweep", False))
RUN_MALICIOUS_FRACTION_SWEEP = bool(attack_module_config.get("run_malicious_fraction_sweep", False))
RUN_ALGORITHM_COMPARISON = bool(attack_module_config.get("run_algorithm_comparison", False))
ATTACK_RECIPE_OPTIONS = {"pgd_default", "fgsm_default", "random_noise"}
ATTACK_RECIPE_SWEEP_RECIPES = [
    str(recipe)
    for recipe in attack_module_config.get(
        "attack_recipe_sweep_recipes",
        ["random_noise", "fgsm_default", "pgd_default"],
    )
]
MALICIOUS_FRACTION_GRID = [float(frac) for frac in attack_module_config.get("malicious_fraction_grid", [0.0, 0.05, 0.1, 0.2])]
ALGORITHM_COMPARISON_ALGORITHMS = _canonical_algorithm_sequence(
    attack_module_config.get("algorithm_comparison_algorithms", AVAILABLE_ALGORITHMS)
)
ALGORITHM_COMPARISON_ATTACK_RECIPE = str(attack_module_config.get("algorithm_comparison_attack_recipe", "pgd_default"))
set_seed(global_config.get("seed", 42))


def get_device(preferred: str | None = None) -> torch.device:
    choice = preferred if preferred is not None else global_config.get("device")
    if isinstance(choice, str):
        if choice.startswith("cuda") and not torch.cuda.is_available():
            choice = "cpu"
        return torch.device(choice)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = get_device()
global_config["device"] = str(DEVICE)

ARTIFACT_DIR = Path(artifact_config.get("dir", "artifacts"))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def artifact_path(key: str, default_filename: str) -> Path:
    return ARTIFACT_DIR / artifact_config.get(key, default_filename)


def _as_int(value, path: str, issues: list[str], default: int = 0) -> int:
    try:
        return int(value)
    except (TypeError, ValueError):
        issues.append(f"{path} must be an integer.")
        return default


def _as_float(value, path: str, issues: list[str], default: float = 0.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        issues.append(f"{path} must be numeric.")
        return default


def _validate_fed_config(name: str, fed_cfg: dict, issues: list[str]) -> None:
    required = (
        "fraction_clients",
        "num_clients",
        "num_rounds",
        "num_epochs",
        "batch_size",
        "global_stepsize",
        "local_stepsize",
        "criterion",
    )
    for key in required:
        if key not in fed_cfg:
            issues.append(f"algorithms.{name}.fed_config.{key} is required.")

    fraction = _as_float(fed_cfg.get("fraction_clients"), f"algorithms.{name}.fed_config.fraction_clients", issues)
    num_clients = _as_int(fed_cfg.get("num_clients"), f"algorithms.{name}.fed_config.num_clients", issues)
    num_rounds = _as_int(fed_cfg.get("num_rounds"), f"algorithms.{name}.fed_config.num_rounds", issues)
    num_epochs = _as_int(fed_cfg.get("num_epochs"), f"algorithms.{name}.fed_config.num_epochs", issues)
    batch_size = _as_int(fed_cfg.get("batch_size"), f"algorithms.{name}.fed_config.batch_size", issues)
    global_stepsize = _as_float(fed_cfg.get("global_stepsize"), f"algorithms.{name}.fed_config.global_stepsize", issues)
    local_stepsize = _as_float(fed_cfg.get("local_stepsize"), f"algorithms.{name}.fed_config.local_stepsize", issues)

    if not 0.0 < fraction <= 1.0:
        issues.append(f"algorithms.{name}.fed_config.fraction_clients must be in (0, 1].")
    if num_clients <= 0:
        issues.append(f"algorithms.{name}.fed_config.num_clients must be positive.")
    if num_rounds <= 0:
        issues.append(f"algorithms.{name}.fed_config.num_rounds must be positive.")
    if num_epochs <= 0:
        issues.append(f"algorithms.{name}.fed_config.num_epochs must be positive.")
    if batch_size <= 0:
        issues.append(f"algorithms.{name}.fed_config.batch_size must be positive.")
    if global_stepsize <= 0:
        issues.append(f"algorithms.{name}.fed_config.global_stepsize must be positive.")
    if local_stepsize <= 0:
        issues.append(f"algorithms.{name}.fed_config.local_stepsize must be positive.")


def validate_attack_config() -> None:
    issues: list[str] = []
    if SELECTED_ALGORITHM not in SUPPORTED_ALGORITHMS:
        issues.append(
            f"attack_module.selected_algorithm={RAW_SELECTED_ALGORITHM!r} is unsupported. "
            f"Available algorithms: {SUPPORTED_ALGORITHMS}."
        )
    if not SELECTED_ALG_CONFIG:
        issues.append(f"algorithms.{SELECTED_ALGORITHM} must be configured.")

    for configured_name, configured in alg_configs.items():
        try:
            canonical_name = canonical_algorithm_name(configured_name)
        except KeyError:
            issues.append(
                f"algorithms.{configured_name} is not registered. "
                f"Available algorithms: {SUPPORTED_ALGORITHMS}."
            )
            continue
        _validate_fed_config(canonical_name, (configured or {}).get("fed_config", {}), issues)

    fed_cfg = DEFAULT_FED_CONFIG
    attack_cfg = attack_defaults.get("attack", {})
    surrogate_mode = str(surrogate_training_config.get("mode", "centralized")).lower()
    num_rounds = _as_int(fed_cfg.get("num_rounds"), "selected fed_config.num_rounds", issues)
    num_clients = _as_int(fed_cfg.get("num_clients"), "selected fed_config.num_clients", issues)
    start_round = _as_int(attack_defaults.get("start_round", 0), "attack.start_round", issues)
    malicious_fraction = _as_float(attack_defaults.get("malicious_fraction", 0.0), "attack.malicious_fraction", issues)
    num_classes = _as_int(model_config.get("kwargs", {}).get("num_classes", 10), "model_config.kwargs.num_classes", issues, 10)
    if model_config.get("name") != "MobileNetV3Transfer":
        issues.append("model_config.name must be MobileNetV3Transfer for the Module 4 target checkpoint.")
    if num_classes <= 0:
        issues.append("model_config.kwargs.num_classes must be positive.")
    target_label = attack_cfg.get("target_label")
    attack_type = str(attack_cfg.get("type", "pgd")).lower()
    poison_rate = _as_float(attack_cfg.get("poison_rate", 0.0), "attack.attack.poison_rate", issues)
    epsilon = _as_float(attack_cfg.get("epsilon", 0.0), "attack.attack.epsilon", issues)
    step_size = _as_float(attack_cfg.get("step_size", 0.0), "attack.attack.step_size", issues)
    iters = _as_int(attack_cfg.get("iters", 0), "attack.attack.iters", issues)

    if surrogate_mode != "centralized":
        issues.append("surrogate_training.mode must be 'centralized' so the saved surrogate checkpoint is the attack source.")
    if num_rounds > 0 and start_round > num_rounds:
        issues.append(f"attack.start_round ({start_round}) must not be after the final round ({num_rounds}).")
    if not 0.0 <= malicious_fraction <= 1.0:
        issues.append("attack.malicious_fraction must be in [0, 1].")
    if attack_type not in ATTACK_FUNCTIONS:
        issues.append(f"attack.attack.type={attack_type!r} is unsupported. Available attacks: {sorted(ATTACK_FUNCTIONS)}.")
    if target_label is None:
        issues.append("attack.attack.target_label must be set.")
    elif not 0 <= int(target_label) < num_classes:
        issues.append(f"attack.attack.target_label must be in [0, {num_classes - 1}].")
    if not 0.0 <= poison_rate <= 1.0:
        issues.append("attack.attack.poison_rate must be in [0, 1].")
    if epsilon <= 0:
        issues.append("attack.attack.epsilon must be positive.")
    if step_size <= 0:
        issues.append("attack.attack.step_size must be positive.")
    if attack_type == "pgd" and iters <= 0:
        issues.append("attack.attack.iters must be positive for PGD.")
    if attack_type in {"pgd", "fgsm"} and epsilon > 0 and step_size > epsilon:
        issues.append("attack.attack.step_size must not exceed epsilon for PGD/FGSM recipes.")
    for frac in MALICIOUS_FRACTION_GRID:
        if not 0.0 <= frac <= 1.0:
            issues.append(f"attack_module.malicious_fraction_grid contains invalid fraction {frac}.")

    if RUN_ATTACK_RECIPE_SWEEP and not ATTACK_RECIPE_SWEEP_RECIPES:
        issues.append("attack_module.attack_recipe_sweep_recipes must include at least one recipe when run_attack_recipe_sweep is true.")
    for recipe in ATTACK_RECIPE_SWEEP_RECIPES:
        if recipe not in ATTACK_RECIPE_OPTIONS:
            issues.append(
                "attack_module.attack_recipe_sweep_recipes contains unsupported recipe "
                f"{recipe!r}. Available recipes: {sorted(ATTACK_RECIPE_OPTIONS)}."
            )
    if ALGORITHM_COMPARISON_ATTACK_RECIPE not in ATTACK_RECIPE_OPTIONS:
        issues.append(
            "attack_module.algorithm_comparison_attack_recipe must be one of "
            f"{sorted(ATTACK_RECIPE_OPTIONS)}."
        )
    if RUN_ALGORITHM_COMPARISON and len(ALGORITHM_COMPARISON_ALGORITHMS) < 2:
        issues.append("attack_module.algorithm_comparison_algorithms must include at least two algorithms when run_algorithm_comparison is true.")
    for alg_name in ALGORITHM_COMPARISON_ALGORITHMS:
        if alg_name not in SUPPORTED_ALGORITHMS:
            issues.append(
                f"attack_module.algorithm_comparison_algorithms contains unsupported algorithm {alg_name!r}. "
                f"Available algorithms: {SUPPORTED_ALGORITHMS}."
            )
            continue
        alg_conf = _algorithm_config(alg_name)
        if not alg_conf:
            issues.append(f"algorithms.{alg_name} must be configured for algorithm comparison.")
            continue
        alg_rounds = _as_int(alg_conf.get("fed_config", {}).get("num_rounds"), f"algorithms.{alg_name}.fed_config.num_rounds", issues)
        if RUN_ALGORITHM_COMPARISON and alg_rounds > 0 and start_round > alg_rounds:
            issues.append(
                f"attack.start_round ({start_round}) must not be after the final round "
                f"for compared algorithm {alg_name} ({alg_rounds})."
            )

    try:
        select_malicious_client_ids(attack_defaults, num_clients)
    except Exception as exc:
        issues.append(f"Malicious-client selection is invalid: {exc}")

    if issues:
        raise ValueError("Attack-module config validation failed:\n" + "\n".join(issues))

    print(
        "Attack module config ready: "
        f"device={DEVICE}, algorithm={SELECTED_ALGORITHM}, rounds={num_rounds}, "
        f"attack starts at round {start_round}, malicious_fraction={malicious_fraction}, "
        f"epsilon={epsilon:.5f}, surrogate_profile={SURROGATE_PROFILE}, "
        f"attack_recipe_sweep={RUN_ATTACK_RECIPE_SWEEP}, "
        f"fraction_sweep={RUN_MALICIOUS_FRACTION_SWEEP}, "
        f"algorithm_comparison={RUN_ALGORITHM_COMPARISON}"
    )


validate_attack_config()
RESOLVED_MALICIOUS_CLIENT_IDS = select_malicious_client_ids(
    attack_defaults,
    int(DEFAULT_FED_CONFIG["num_clients"]),
)

config_snapshot = deepcopy(CONFIG)
config_snapshot.setdefault("global_config", {})["resolved_device"] = str(DEVICE)
config_snapshot.setdefault("attack_module", {})["selected_algorithm"] = SELECTED_ALGORITHM
config_snapshot.setdefault("attack_module", {})["available_algorithms"] = AVAILABLE_ALGORITHMS
config_snapshot.setdefault("attack_module", {})["selected_fed_config_effective"] = deepcopy(DEFAULT_FED_CONFIG)
config_snapshot.setdefault("attack_module", {})["selected_optim_config_effective"] = deepcopy(DEFAULT_OPTIM_CONFIG)
config_snapshot.setdefault("attack_module", {})["attack_recipe_sweep_recipes"] = list(ATTACK_RECIPE_SWEEP_RECIPES)
config_snapshot.setdefault("attack", {})["resolved_malicious_client_ids"] = list(RESOLVED_MALICIOUS_CLIENT_IDS)
config_snapshot["threat_model"] = {
    "target_model": "MobileNetV3Transfer",
    "surrogate_model": "MobileNetV2Transfer",
    "target_checkpoint_source": "train_v3.ipynb",
    "surrogate_checkpoint_source": "train_surrogate.ipynb",
    "clean_and_attacked_fl_initialization": "MobileNetV3Transfer target checkpoint",
    "poison_generation_gradient_source": "MobileNetV2Transfer surrogate checkpoint",
    "target_gradients_used_for_poison_generation": False,
}
config_snapshot.setdefault("surrogate_training", {})["selected_profile"] = SURROGATE_PROFILE
config_snapshot["surrogate_training_effective"] = deepcopy(surrogate_training_config)
config_path = artifact_path("config_snapshot", "module4_attack_config_used.json")
with config_path.open("w") as f:
    json.dump(config_snapshot, f, indent=2)

print("Loaded config from", CONFIG_PATH.resolve())
print("Available algorithms:", AVAILABLE_ALGORITHMS)
print(f"Saved attack config snapshot to {config_path.resolve()}")


## 3. Load Target, Surrogate, and Evaluation Data

Load the centralized V3 target checkpoint, the centralized V2 surrogate checkpoint, and a shared evaluation loader. The FL runs later start from `target_state`, and malicious clients receive the same saved surrogate checkpoint unless local fine-tuning is explicitly enabled in config. Poison generation is black-box with respect to the V3 target: only the MobileNetV2 surrogate supplies attack gradients.


In [ ]:
TARGET_MODEL_NAME = "MobileNetV3Transfer"
SURROGATE_MODEL_NAME = "MobileNetV2Transfer"
TARGET_STATE_PREFIX = "v3model"
SURROGATE_STATE_PREFIX = "v2model"


def _resolve_existing_artifact(key: str, default_filename: str, legacy_filenames: list[str] | None = None) -> Path:
    path = artifact_path(key, default_filename)
    if path.exists():
        return path
    for filename in legacy_filenames or []:
        legacy_path = ARTIFACT_DIR / filename
        if legacy_path.exists():
            print(f"Using legacy artifact name for {key}: {legacy_path}")
            return legacy_path
    return path


def _normalise_state_dict_keys(state: dict) -> dict[str, torch.Tensor]:
    if not isinstance(state, dict) or not state:
        raise ValueError("Checkpoint did not contain a non-empty state dict.")
    if not all(isinstance(key, str) for key in state):
        raise ValueError("Checkpoint state dict keys must be strings.")
    if all(key.startswith("module.") for key in state):
        return {key.removeprefix("module."): value for key, value in state.items()}
    return state


def _load_state_dict(path: Path) -> dict[str, torch.Tensor]:
    try:
        state = torch.load(path, map_location="cpu")
    except Exception as exc:
        raise RuntimeError(f"Could not load checkpoint at {path}.") from exc
    if isinstance(state, dict) and "model_state" in state:
        state = state["model_state"]
    elif isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    return _normalise_state_dict_keys(state)


def _checkpoint_family(state: dict[str, torch.Tensor]) -> str:
    keys = list(state)
    if any(key.startswith(f"{TARGET_STATE_PREFIX}.") for key in keys):
        return TARGET_MODEL_NAME
    if any(key.startswith(f"{SURROGATE_STATE_PREFIX}.") for key in keys):
        return SURROGATE_MODEL_NAME
    return "unknown"


def _checkpoint_num_classes(state: dict[str, torch.Tensor], prefix: str) -> int | None:
    classifier_weights = []
    for key, value in state.items():
        if (
            key.startswith(f"{prefix}.classifier.")
            and key.endswith(".weight")
            and torch.is_tensor(value)
            and value.ndim == 2
        ):
            classifier_weights.append((key, int(value.shape[0])))
    if not classifier_weights:
        return None
    return sorted(classifier_weights)[-1][1]


def _validate_checkpoint_summary(
    summary: dict,
    expected_model: str,
    expected_num_classes: int,
    summary_path: Path,
) -> list[str]:
    if not summary:
        return []
    issues = []
    recorded_model = summary.get("model") or summary.get("model_name")
    if recorded_model and recorded_model != expected_model:
        issues.append(
            f"{summary_path} records model={recorded_model!r}, expected {expected_model!r}."
        )
    recorded_kwargs = summary.get("model_kwargs")
    if isinstance(recorded_kwargs, dict) and "num_classes" in recorded_kwargs:
        recorded_classes = int(recorded_kwargs["num_classes"])
        if recorded_classes != expected_num_classes:
            issues.append(
                f"{summary_path} records num_classes={recorded_classes}, "
                f"expected {expected_num_classes}."
            )
    return issues


def _checked_model_from_checkpoint(
    *,
    path: Path,
    state: dict[str, torch.Tensor],
    model_cls,
    expected_model: str,
    expected_prefix: str,
    kwargs: dict,
    summary: dict,
    summary_path: Path,
) -> torch.nn.Module:
    issues = []
    expected_num_classes = int(kwargs.get("num_classes", num_classes))
    family = _checkpoint_family(state)
    if family != expected_model:
        issues.append(
            f"State dict looks like {family!r}; expected {expected_model!r} keys "
            f"with prefix {expected_prefix!r}."
        )
    checkpoint_classes = _checkpoint_num_classes(state, expected_prefix)
    if checkpoint_classes is None:
        issues.append(
            f"Could not infer class count from {expected_model} classifier weights in {path}."
        )
    elif checkpoint_classes != expected_num_classes:
        issues.append(
            f"Checkpoint classifier has {checkpoint_classes} output classes; "
            f"config expects {expected_num_classes}."
        )
    issues.extend(
        _validate_checkpoint_summary(summary, expected_model, expected_num_classes, summary_path)
    )

    model = model_cls(**kwargs).to(DEVICE)
    try:
        model.load_state_dict(state, strict=True)
    except RuntimeError as exc:
        issues.append(f"Strict state_dict load failed: {exc}")

    if issues:
        issue_text = "\n".join(f"- {issue}" for issue in issues)
        raise RuntimeError(
            f"{expected_model} checkpoint at {path} is incompatible with "
            f"attack_module_config.yaml:\n{issue_text}"
        )

    model.eval()
    return model


def _target_model_kwargs() -> dict:
    if model_config.get("name") != TARGET_MODEL_NAME:
        raise ValueError(
            f"model_config.name must be {TARGET_MODEL_NAME!r}; "
            f"got {model_config.get('name')!r}."
        )
    kwargs = deepcopy(model_config.get("kwargs", {}))
    kwargs["pretrained"] = False
    kwargs["num_classes"] = int(kwargs.get("num_classes", num_classes))
    return kwargs


def _checkpoint_model_config() -> dict:
    cfg = deepcopy(model_config)
    cfg["name"] = TARGET_MODEL_NAME
    cfg["kwargs"] = _target_model_kwargs()
    return cfg


def build_target_model_from_checkpoint() -> torch.nn.Module:
    target = MobileNetV3Transfer(**_target_model_kwargs()).to(DEVICE)
    try:
        target.load_state_dict(target_state, strict=True)
    except RuntimeError as exc:
        raise RuntimeError(
            f"Validated target checkpoint no longer matches {TARGET_MODEL_NAME}: "
            f"{target_checkpoint_path}"
        ) from exc
    target.eval()
    return target


target_checkpoint_path = _resolve_existing_artifact(
    "target_checkpoint",
    "module4_v3_target.pt",
    legacy_filenames=["module4_fedavg_target.pt"],
)
target_summary_path = artifact_path("target_metrics", "module4_target_training.json")
surrogate_checkpoint_path = artifact_path("surrogate_checkpoint", "module4_surrogate.pt")
surrogate_summary_path = artifact_path("surrogate_metrics", "module4_surrogate.json")

missing = [
    path
    for path in [target_checkpoint_path, surrogate_checkpoint_path, surrogate_summary_path]
    if not path.exists()
]
if missing:
    missing_text = "\n".join(str(path) for path in missing)
    raise FileNotFoundError(
        "Run train_v3.ipynb and train_surrogate.ipynb before this attack notebook. "
        f"Missing required artifact(s):\n{missing_text}"
    )

with surrogate_summary_path.open("r") as f:
    surrogate_summary = json.load(f)
if target_summary_path.exists():
    with target_summary_path.open("r") as f:
        target_summary = json.load(f)
else:
    target_summary = {}
    print(f"Target summary not found at {target_summary_path}; continuing with checkpoint-only target state.")

num_classes = int(model_config.get("kwargs", {}).get("num_classes", 10))
target_state = _load_state_dict(target_checkpoint_path)
surrogate_state = _load_state_dict(surrogate_checkpoint_path)

target_model = _checked_model_from_checkpoint(
    path=target_checkpoint_path,
    state=target_state,
    model_cls=MobileNetV3Transfer,
    expected_model=TARGET_MODEL_NAME,
    expected_prefix=TARGET_STATE_PREFIX,
    kwargs=_target_model_kwargs(),
    summary=target_summary,
    summary_path=target_summary_path,
)
surrogate_model = _checked_model_from_checkpoint(
    path=surrogate_checkpoint_path,
    state=surrogate_state,
    model_cls=MobileNetV2Transfer,
    expected_model=SURROGATE_MODEL_NAME,
    expected_prefix=SURROGATE_STATE_PREFIX,
    kwargs={"pretrained": False, "num_classes": num_classes},
    summary=surrogate_summary,
    summary_path=surrogate_summary_path,
)

config_snapshot["artifact_paths"] = {
    "target_checkpoint_path": str(target_checkpoint_path),
    "surrogate_checkpoint_path": str(surrogate_checkpoint_path),
    "target_summary_path": str(target_summary_path),
    "surrogate_summary_path": str(surrogate_summary_path),
}
config_snapshot["checkpoint_validation"] = {
    "target_model": TARGET_MODEL_NAME,
    "target_checkpoint_num_classes": _checkpoint_num_classes(target_state, TARGET_STATE_PREFIX),
    "target_config_num_classes": num_classes,
    "surrogate_model": SURROGATE_MODEL_NAME,
    "surrogate_checkpoint_num_classes": _checkpoint_num_classes(surrogate_state, SURROGATE_STATE_PREFIX),
    "surrogate_config_num_classes": num_classes,
    "target_gradients_used_for_poison_generation": False,
}
with config_path.open("w") as f:
    json.dump(config_snapshot, f, indent=2)

ATTACK_EVAL_SUBSET = str(attack_module_config.get("eval_subset", data_config.get("eval_subset", "attack_eval")))
attack_train_dataset, full_attack_val_dataset = create_data(
    data_config["dataset_path"],
    data_config["dataset_name"],
)
attack_test_dataset = select_validation_subset(
    full_attack_val_dataset,
    data_config.get("validation_split"),
    ATTACK_EVAL_SUBSET,
)
attack_eval_loader = DataLoader(
    attack_test_dataset,
    batch_size=int(attack_module_config.get("eval_batch_size", 512)),
    shuffle=False,
    drop_last=False,
    num_workers=int(attack_module_config.get("num_workers", 0)),
    pin_memory=bool(attack_module_config.get("pin_memory", DEVICE.type == "cuda")),
)
attack_criterion = nn.CrossEntropyLoss().to(DEVICE)
target_eval_loss, target_eval_acc = evaluate_fn(attack_eval_loader, target_model, attack_criterion, DEVICE)
baseline_results: dict[str, dict] = {}

print(f"Loaded MobileNetV3 target checkpoint from {target_checkpoint_path.resolve()}")
print(f"Loaded MobileNetV2 surrogate checkpoint from {surrogate_checkpoint_path.resolve()}")
print("Checkpoint validation passed: V3 target initializes FL; V2 surrogate crafts poisons.")
print(
    f"Attack evaluation subset: {ATTACK_EVAL_SUBSET} "
    f"({len(attack_test_dataset)} of {len(full_attack_val_dataset)} validation examples)"
)
print(f"Target checkpoint held-out evaluation: loss={target_eval_loss:.4f}, accuracy={target_eval_acc:.2f}%")
print(
    "Surrogate recorded validation accuracy: "
    f"{float(surrogate_summary.get('validation_accuracy', surrogate_summary.get('test_accuracy', 0.0))):.2f}%"
)


## 4. FL Helpers

These helpers run clean or attacked FL from the centralized V3 checkpoint. The clean and attacked runs share the same initialization, client split, rounds, and summary fields.


In [ ]:
def _clear_cuda_cache() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _to_builtin(value):
    if isinstance(value, torch.Tensor):
        value = value.detach().cpu().item()
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (int, float, str, bool)) or value is None:
        return value
    return float(value)


def _json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, torch.Tensor):
        if value.numel() == 1:
            return _to_builtin(value)
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_json_safe(v) for v in value]
    if callable(value):
        return getattr(value, "__name__", repr(value))
    if isinstance(value, (int, float, str, bool)) or value is None:
        return value
    return str(value)


def _set_server_initial_state(server, state: dict[str, torch.Tensor]) -> None:
    server.global_model.load_state_dict(state)
    server.global_model.to(server.device)
    server.x = server.global_model


def train_fl_server(algorithm_name: str | None = None, attack_cfg: dict | None = None):
    alg_name = canonical_algorithm_name(algorithm_name or SELECTED_ALGORITHM)
    alg_conf = _algorithm_config(alg_name)
    if not alg_conf:
        raise ValueError(f"Algorithm {alg_name!r} not found in attack_module_config.yaml.")

    fed_cfg = deepcopy(alg_conf["fed_config"])
    fed_cfg["algorithm"] = alg_name
    optim_cfg = deepcopy(alg_conf.get("optim_config", {}))
    attack_cfg = deepcopy(attack_cfg or {"malicious_fraction": 0.0})

    logs_dir = os.path.join("Logs", alg_name, str(data_config.get("non_iid_per", 0.0)))
    os.makedirs(logs_dir, exist_ok=True)
    set_logger(os.path.join(logs_dir, "log.txt"))

    server_cls = get_algorithm_server_class(alg_name)
    server = server_cls(
        _checkpoint_model_config(),
        global_config,
        data_config,
        fed_cfg,
        optim_cfg,
        attack_cfg,
    )
    server.setup()
    _set_server_initial_state(server, target_state)
    print(f"Initialized {alg_name} global model from {target_checkpoint_path.name}")
    server.train()
    return server


def _target_label_from_attack_config(attack_cfg: dict | None) -> int | None:
    attack_params = (attack_cfg or {}).get("attack", {})
    target_label = attack_params.get("target_label")
    return int(target_label) if target_label is not None else None


def summarise_server(
    server,
    target_label: int | None = None,
    attack_cfg: dict | None = None,
    recipe_name: str | None = None,
) -> dict:
    model = getattr(server, "global_model", getattr(server, "x", None))
    loss, acc = evaluate_fn(server.test_loader, model, server.criterion, server.device)
    raw_history = server.results if hasattr(server, "results") else {}
    history = {}
    for key, values in raw_history.items():
        if isinstance(values, (list, tuple)):
            history[key] = [_json_safe(value) for value in values]

    algorithm_name = str(server.fed_config.get("algorithm", SELECTED_ALGORITHM))
    final_metrics = {
        "loss": float(loss),
        "accuracy": float(acc),
    }
    summary = {
        "algorithm": algorithm_name,
        "selected_device": str(server.device),
        "initial_checkpoint": str(target_checkpoint_path),
        "target_checkpoint_path": str(target_checkpoint_path),
        "surrogate_checkpoint_path": str(surrogate_checkpoint_path),
        "target_model": TARGET_MODEL_NAME,
        "surrogate_model": SURROGATE_MODEL_NAME,
        "poison_generation_gradient_source": "surrogate_model",
        "target_gradients_used_for_poison_generation": False,
        "config_snapshot_path": str(config_path),
        "fl_settings": _json_safe(server.fed_config),
        "optim_config": _json_safe(server.optim_config),
        "attack_recipe_name": recipe_name,
        "attack_recipe": _json_safe(attack_cfg or {}),
        "final_loss": float(loss),
        "final_accuracy": float(acc),
        "final_metrics": final_metrics,
        "history": history,
        "malicious_client_ids": list(getattr(server, "malicious_client_ids", [])),
        "run_provenance": {
            "config_path": str(CONFIG_PATH),
            "config_snapshot_path": str(config_path),
            "selected_algorithm": algorithm_name,
            "selected_device": str(server.device),
            "target_checkpoint_path": str(target_checkpoint_path),
            "surrogate_checkpoint_path": str(surrogate_checkpoint_path),
            "target_model": TARGET_MODEL_NAME,
            "surrogate_model": SURROGATE_MODEL_NAME,
            "poison_generation_gradient_source": "surrogate_model",
            "target_gradients_used_for_poison_generation": False,
            "initialized_from_target_checkpoint": True,
        },
    }
    if target_label is not None:
        global_target_label_asr = float(
            target_label_prediction_rate(
                server.test_loader,
                model,
                target_label,
                server.device,
                exclude_true_target_label=True,
            )
        )
        summary["global_target_label"] = int(target_label)
        summary["global_target_label_asr"] = global_target_label_asr
        summary["final_global_target_label_asr"] = global_target_label_asr
        final_metrics["global_target_label_asr"] = global_target_label_asr

    for metric in (
        "surrogate_poison_success_rate",
        "global_target_label_asr",
        "poisoned_examples",
        "candidate_examples",
        "sampled_malicious_clients",
    ):
        values = history.get(metric, [])
        if values:
            summary[f"final_{metric}"] = _json_safe(values[-1])
            final_metrics[metric] = _json_safe(values[-1])
    return summary


def run_one_fl(algorithm_name: str | None = None, attack_cfg: dict | None = None, recipe_name: str | None = None) -> dict:
    server = train_fl_server(algorithm_name=algorithm_name, attack_cfg=attack_cfg)
    summary = summarise_server(
        server,
        target_label=_target_label_from_attack_config(attack_cfg),
        attack_cfg=attack_cfg,
        recipe_name=recipe_name,
    )
    del server
    _clear_cuda_cache()
    return summary


## 5. Attack Configuration

Extract shared attack settings and define the clean, random-noise, FGSM, and PGD recipes.


In [ ]:
SURROGATE_CFG = surrogate_training_config
SURROGATE_CLIENT_ID = int(SURROGATE_CFG.get("client_id", 0))
SURROGATE_BATCH_SIZE = int(SURROGATE_CFG.get("batch_size", DEFAULT_FED_CONFIG.get("batch_size", 64)))
ATTACK_RAW = attack_defaults
ATTACK_BASE = ATTACK_RAW.get("attack", {})
SURROGATE_BASE = ATTACK_RAW.get("surrogate", {})
ATTACK_SEED = ATTACK_RAW.get("seed", global_config.get("seed", 42))
BASE_MALICIOUS_FRACTION = ATTACK_RAW.get("malicious_fraction", 0.0)
ATTACK_EPSILON = float(ATTACK_BASE.get("epsilon", 0.03137255))


def _extract_attack_params(overrides: dict | None = None) -> dict:
    params = {
        "type": ATTACK_BASE.get("type", "pgd"),
        "poison_rate": ATTACK_BASE.get("poison_rate", 0.0),
        "target_label": ATTACK_BASE.get("target_label", 0),
        "epsilon": ATTACK_EPSILON,
        "step_size": ATTACK_BASE.get("step_size", 0.00784314),
        "iters": ATTACK_BASE.get("iters", 10),
        "criterion": ATTACK_BASE.get("criterion", "torch.nn.CrossEntropyLoss"),
    }
    schedule = ATTACK_BASE.get("poison_rate_schedule")
    if schedule:
        params["poison_rate_schedule"] = schedule
    if overrides:
        params.update(overrides)
    return params


def _extract_surrogate_params(overrides: dict | None = None) -> dict:
    local_finetune_epochs = int(
        SURROGATE_CFG.get(
            "local_finetune_epochs",
            SURROGATE_BASE.get("finetune_epochs", 0),
        )
    )
    configured_checkpoint = (
        SURROGATE_BASE.get("checkpoint")
        or SURROGATE_BASE.get("checkpoint_path")
        or str(surrogate_checkpoint_path)
    )
    params = {
        "checkpoint": str(surrogate_checkpoint_path),
        "configured_checkpoint": configured_checkpoint,
        "checkpoint_source": SURROGATE_BASE.get("checkpoint_source", "train_surrogate.ipynb"),
        "poison_generation_model": SURROGATE_MODEL_NAME,
        "poison_generation_gradient_source": "surrogate_model",
        "target_gradients_used_for_poison_generation": False,
        "training_profile": SURROGATE_PROFILE,
        "pretrained": False,
        "finetune_epochs": local_finetune_epochs,
        "lr": SURROGATE_CFG.get("learning_rate", SURROGATE_BASE.get("learning_rate", SURROGATE_BASE.get("lr", 1e-3))),
        "weight_decay": SURROGATE_CFG.get("weight_decay", SURROGATE_BASE.get("weight_decay", 0.0)),
        "batch_size": SURROGATE_BATCH_SIZE,
        "client_id": SURROGATE_CLIENT_ID,
        "pool_size": SURROGATE_CFG.get("pool_size", SURROGATE_BASE.get("pool_size", 1)),
        "freeze_backbone": SURROGATE_CFG.get("freeze_backbone", SURROGATE_BASE.get("freeze_backbone", False)),
        "augment": SURROGATE_CFG.get("augment", SURROGATE_BASE.get("augment", False)),
        "early_stop_patience": SURROGATE_CFG.get("early_stop_patience", SURROGATE_BASE.get("early_stop_patience", 0)),
        "num_classes": SURROGATE_CFG.get("num_classes", SURROGATE_BASE.get("num_classes", num_classes)),
    }
    if overrides:
        params.update(overrides)
    return params


def build_attack_config(
    *,
    attack_overrides: dict | None = None,
    surrogate_overrides: dict | None = None,
    malicious_fraction: float | None = None,
    seed: int | None = None,
) -> dict:
    resolved_fraction = BASE_MALICIOUS_FRACTION if malicious_fraction is None else malicious_fraction
    selection_cfg = deepcopy(ATTACK_RAW.get("malicious_client_selection", {}))
    if float(resolved_fraction) == 0.0:
        selection_cfg = {"mode": "none"}

    cfg = {
        "seed": seed if seed is not None else ATTACK_SEED,
        "malicious_fraction": resolved_fraction,
        "malicious_client_selection": selection_cfg,
        "attack": _extract_attack_params(attack_overrides),
        "surrogate": _extract_surrogate_params(surrogate_overrides),
    }
    for key in ("start_round", "malicious_client_ids", "malicious_selection_mode"):
        if key in ATTACK_RAW:
            cfg[key] = deepcopy(ATTACK_RAW[key])
    return cfg


ATTACK_RECIPES = {
    "clean": build_attack_config(malicious_fraction=0.0, attack_overrides={"poison_rate": 0.0}),
    "pgd_default": build_attack_config(),
    "fgsm_default": build_attack_config(attack_overrides={"type": "fgsm", "step_size": ATTACK_EPSILON, "iters": 1}),
    "random_noise": build_attack_config(attack_overrides={"type": "random_noise", "step_size": ATTACK_EPSILON, "iters": 1}),
}


## 6. Malicious Client and Attack Helpers

These helpers craft adversarial batches with the loaded surrogate checkpoint and run attack recipes through the selected FL algorithm from the loaded V3 checkpoint.


In [ ]:
def select_attack_fn(name: str) -> AttackFn:
    key = name.lower()
    if key not in ATTACK_FUNCTIONS:
        raise KeyError(f"Unknown attack function '{name}'. Available: {sorted(ATTACK_FUNCTIONS)}")
    return ATTACK_FUNCTIONS[key]


def _attach_attack_callable(cfg: dict) -> dict:
    cfg_copy = deepcopy(cfg)
    attack_params = cfg_copy.setdefault("attack", {})
    attack_type = attack_params.get("type", "pgd")
    if "callable" not in attack_params:
        attack_params["callable"] = select_attack_fn(attack_type)
    return cfg_copy


def attack_label(recipe_name: str) -> str:
    return {
        "random_noise": "Random",
        "fgsm_default": "FGSM",
        "pgd_default": "PGD",
        "clean": "Clean",
    }.get(recipe_name, recipe_name)


def make_malicious_client(
    attack_config: dict,
    *,
    local_loader=None,
    surrogate=None,
    num_epochs: int = 0,
    lr: float | None = None,
):
    if local_loader is None:
        local_loader = attack_eval_loader
    if surrogate is None:
        surrogate = surrogate_model

    client = MaliciousClient(
        client_id=SURROGATE_CLIENT_ID,
        local_data=local_loader,
        device=DEVICE,
        num_epochs=num_epochs,
        criterion=nn.CrossEntropyLoss().to(DEVICE),
        lr=lr if lr is not None else DEFAULT_FED_CONFIG.get("local_stepsize", 0.003),
        attack_config=attack_config,
    )
    client.surrogate = surrogate.to(DEVICE)
    client.surrogate.eval()
    return client


def craft_adversarial_batch(client: MaliciousClient, inputs: torch.Tensor, labels: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    target_label = int(client.attack_params.get("target_label", 0))
    target_labels = torch.full_like(labels, target_label)
    client.surrogate.eval()
    adv = client.perform_attack(inputs, target_labels)
    return adv, target_labels


def _next_eval_batch(batch_size: int, target_label: int | None = None):
    if target_label is None:
        configured_target = ATTACK_BASE.get("target_label")
        target_label = int(configured_target) if configured_target is not None else None
    input_chunks = []
    label_chunks = []
    collected = 0
    for inputs, labels in attack_eval_loader:
        inputs = inputs.to(DEVICE).float()
        labels = labels.to(DEVICE).long()
        if target_label is not None:
            mask = labels != int(target_label)
            inputs = inputs[mask]
            labels = labels[mask]
        if labels.numel() == 0:
            continue
        input_chunks.append(inputs)
        label_chunks.append(labels)
        collected += int(labels.size(0))
        if collected >= batch_size:
            break
    if not label_chunks or collected < batch_size:
        target_msg = "" if target_label is None else f" with true label != {target_label}"
        raise ValueError(
            f"Could not collect {batch_size} attack-evaluation examples{target_msg}; "
            f"collected {collected}."
        )
    return torch.cat(input_chunks, dim=0)[:batch_size], torch.cat(label_chunks, dim=0)[:batch_size]


def _attack_metrics(model: torch.nn.Module, inputs: torch.Tensor, labels: torch.Tensor, adv_inputs: torch.Tensor, adv_labels: torch.Tensor) -> dict:
    model.eval()
    with torch.no_grad():
        clean_logits = model(inputs)
        clean_preds = clean_logits.argmax(dim=1)
        adv_logits = model(adv_inputs)
        adv_preds = adv_logits.argmax(dim=1)
    clean_acc = (clean_preds == labels).float().mean().item() * 100.0
    robust_acc = (adv_preds == labels).float().mean().item() * 100.0
    target_label_success_rate = (adv_preds == adv_labels).float().mean().item() * 100.0
    misclassification_rate = (adv_preds != labels).float().mean().item() * 100.0
    linf = pixel_linf_norm(inputs, adv_inputs, normalized=True)
    return {
        "clean_accuracy": clean_acc,
        "robust_accuracy": robust_acc,
        "target_label_success_rate": target_label_success_rate,
        "misclassification_rate": misclassification_rate,
        "mean_linf": linf.mean().item(),
        "max_linf": linf.max().item(),
    }


def evaluate_surrogate_attack(recipe_name: str, *, batch_size: int = 32) -> dict:
    attack_cfg = _attach_attack_callable(ATTACK_RECIPES[recipe_name])
    client = make_malicious_client(attack_cfg, surrogate=surrogate_model, num_epochs=0)
    target_label = int(attack_cfg["attack"].get("target_label", ATTACK_BASE.get("target_label", 0)))
    inputs, labels = _next_eval_batch(batch_size, target_label=target_label)
    adv_inputs, adv_labels = craft_adversarial_batch(client, inputs, labels)
    metrics = _attack_metrics(surrogate_model, inputs, labels, adv_inputs, adv_labels)
    metrics.update({
        "recipe": recipe_name,
        "attack": attack_label(recipe_name),
        "epsilon": float(attack_cfg["attack"].get("epsilon", ATTACK_EPSILON)),
        "surrogate_target_label_success_rate": metrics.pop("target_label_success_rate"),
        "surrogate_misclassification_rate": metrics.pop("misclassification_rate"),
    })
    return metrics


def evaluate_transfer_attack(recipe_name: str, target_model: torch.nn.Module, *, batch_size: int = 32) -> dict:
    attack_cfg = _attach_attack_callable(ATTACK_RECIPES[recipe_name])
    client = make_malicious_client(attack_cfg, surrogate=surrogate_model, num_epochs=0)
    target_label = int(attack_cfg["attack"].get("target_label", ATTACK_BASE.get("target_label", 0)))
    inputs, labels = _next_eval_batch(batch_size, target_label=target_label)
    adv_inputs, adv_labels = craft_adversarial_batch(client, inputs, labels)
    metrics = _attack_metrics(target_model, inputs, labels, adv_inputs, adv_labels)
    metrics.update({
        "recipe": recipe_name,
        "attack": attack_label(recipe_name),
        "epsilon": float(attack_cfg["attack"].get("epsilon", ATTACK_EPSILON)),
        "target_clean_accuracy": metrics.pop("clean_accuracy"),
        "target_robust_accuracy": metrics.pop("robust_accuracy"),
        "target_model_target_label_success_rate": metrics.pop("target_label_success_rate"),
        "surrogate_to_target_transfer_success_rate": metrics.pop("misclassification_rate"),
    })
    return metrics


def run_attack_recipe_on_server(
    recipe_name: str,
    algorithm_name: str | None = None,
    malicious_fraction: float | None = None,
) -> dict:
    alg_name = canonical_algorithm_name(algorithm_name or SELECTED_ALGORITHM)
    attack_cfg = _attach_attack_callable(ATTACK_RECIPES[recipe_name])
    if malicious_fraction is not None:
        attack_cfg = deepcopy(attack_cfg)
        attack_cfg["malicious_fraction"] = malicious_fraction
        if float(malicious_fraction) == 0.0:
            attack_cfg["malicious_client_selection"] = {"mode": "none"}
    summary = run_one_fl(algorithm_name=alg_name, attack_cfg=attack_cfg, recipe_name=recipe_name)
    summary.update({"recipe": recipe_name, "algorithm": alg_name})
    return summary


def sweep_attacks_on_server(
    algorithm_name: str | None = None,
    recipes: list[str] | None = None,
    malicious_fraction: float | None = None,
) -> dict:
    global baseline_results
    alg_name = canonical_algorithm_name(algorithm_name or SELECTED_ALGORITHM)
    recipes = recipes or list(ATTACK_RECIPES)
    results = {}
    for recipe in recipes:
        if recipe == "clean":
            if alg_name not in baseline_results:
                baseline_results[alg_name] = run_attack_recipe_on_server(
                    "clean",
                    algorithm_name=alg_name,
                    malicious_fraction=0.0,
                )
            results[recipe] = deepcopy(baseline_results[alg_name])
            continue
        results[recipe] = run_attack_recipe_on_server(
            recipe,
            algorithm_name=alg_name,
            malicious_fraction=malicious_fraction,
        )
    return results



def _summary_value(summary: dict, *keys: str, default=None):
    final_metrics = summary.get("final_metrics", {}) if isinstance(summary, dict) else {}
    for key in keys:
        if key in summary and summary[key] is not None:
            return summary[key]
        if key.startswith("final_"):
            metric_key = key.removeprefix("final_")
            if metric_key in final_metrics and final_metrics[metric_key] is not None:
                return final_metrics[metric_key]
        if key in final_metrics and final_metrics[key] is not None:
            return final_metrics[key]
    return default


def _total_history_value(summary: dict, key: str, default=0):
    values = summary.get("history", {}).get(key, []) if isinstance(summary, dict) else []
    if values:
        return sum(float(value) for value in values)
    return _summary_value(summary, f"final_{key}", key, default=default)


def build_clean_attacked_summary_row(
    algorithm_name: str,
    results: dict[str, dict],
    *,
    attack_recipe: str = "pgd_default",
    extra: dict | None = None,
) -> dict:
    alg_name = canonical_algorithm_name(algorithm_name)
    clean_summary = results.get("clean", {})
    attack_summary = results.get(attack_recipe, {})
    clean_acc = _summary_value(clean_summary, "final_accuracy", "accuracy")
    attack_acc = _summary_value(attack_summary, "final_accuracy", "accuracy")
    accuracy_drop = (
        float(clean_acc) - float(attack_acc)
        if clean_acc is not None and attack_acc is not None
        else None
    )

    fl_settings = attack_summary.get("fl_settings") or clean_summary.get("fl_settings") or {}
    attack_recipe_cfg = attack_summary.get("attack_recipe", {})
    row = {
        "algorithm": alg_name,
        "attack_recipe": attack_recipe,
        "final_clean_accuracy": clean_acc,
        "final_attacked_accuracy": attack_acc,
        "accuracy_drop": accuracy_drop,
        "surrogate_poison_success_rate": _summary_value(
            attack_summary,
            "final_surrogate_poison_success_rate",
            "surrogate_poison_success_rate",
            default=0.0,
        ),
        "global_target_label_asr": _summary_value(
            attack_summary,
            "final_global_target_label_asr",
            "global_target_label_asr",
            default=0.0,
        ),
        "final_global_target_label_asr": _summary_value(
            attack_summary,
            "final_global_target_label_asr",
            "global_target_label_asr",
            default=0.0,
        ),
        "poisoned_examples": int(_total_history_value(attack_summary, "poisoned_examples", default=0) or 0),
        "malicious_client_ids": attack_summary.get("malicious_client_ids", []),
        "malicious_fraction": attack_recipe_cfg.get("malicious_fraction", BASE_MALICIOUS_FRACTION),
        "num_clients": fl_settings.get("num_clients"),
        "fraction_clients": fl_settings.get("fraction_clients"),
        "num_rounds": fl_settings.get("num_rounds"),
        "num_epochs": fl_settings.get("num_epochs"),
        "batch_size": fl_settings.get("batch_size"),
        "local_stepsize": fl_settings.get("local_stepsize"),
        "global_stepsize": fl_settings.get("global_stepsize"),
        "non_iid_per": data_config.get("non_iid_per"),
        "attack_start_round": attack_recipe_cfg.get("start_round", ATTACK_RAW.get("start_round")),
    }
    if extra:
        row.update(extra)
    return _json_safe(row)


def run_algorithm_comparison(
    algorithms: list[str] | None = None,
    *,
    attack_recipe: str | None = None,
) -> dict:
    recipe = attack_recipe or ALGORITHM_COMPARISON_ATTACK_RECIPE
    alg_names = algorithms or ALGORITHM_COMPARISON_ALGORITHMS
    comparison_results = {}
    rows = []
    for algorithm in alg_names:
        alg_name = canonical_algorithm_name(algorithm)
        print(f"Running algorithm comparison for {alg_name}: clean vs {recipe}.")
        results = sweep_attacks_on_server(
            algorithm_name=alg_name,
            recipes=["clean", recipe],
        )
        comparison_results[alg_name] = results
        rows.append(
            build_clean_attacked_summary_row(
                alg_name,
                results,
                attack_recipe=recipe,
            )
        )

    return {
        "attack_recipe": recipe,
        "algorithms": [row["algorithm"] for row in rows],
        "data_settings": _json_safe(data_config),
        "attack_recipe_config": _json_safe(ATTACK_RECIPES[recipe]),
        "summary_table": rows,
        "results": comparison_results,
    }


def run_attack_recipe_sweep(
    algorithm_name: str | None = None,
    *,
    recipes: list[str] | None = None,
) -> dict:
    alg_name = canonical_algorithm_name(algorithm_name or SELECTED_ALGORITHM)
    recipe_names = recipes or ATTACK_RECIPE_SWEEP_RECIPES
    sweep_results = {}
    rows = []
    for recipe in recipe_names:
        if recipe not in ATTACK_RECIPES or recipe == "clean":
            raise ValueError(
                f"Unsupported FL attack recipe {recipe!r}. "
                f"Available attacked recipes: {sorted(ATTACK_RECIPE_OPTIONS)}."
            )
        print(f"Running attack-recipe sweep for {alg_name}: clean vs {recipe}.")
        results = sweep_attacks_on_server(
            algorithm_name=alg_name,
            recipes=["clean", recipe],
        )
        sweep_results[recipe] = results
        rows.append(
            build_clean_attacked_summary_row(
                alg_name,
                results,
                attack_recipe=recipe,
            )
        )

    return {
        "algorithm": alg_name,
        "attack_recipes": list(recipe_names),
        "data_settings": _json_safe(data_config),
        "fl_settings": _json_safe(_algorithm_config(alg_name).get("fed_config", {})),
        "summary_table": rows,
        "results": sweep_results,
    }


## 7. Poison Generation Dry Run

Before running FL, craft one attack-evaluation batch through the same `MaliciousClient.perform_attack` path used during malicious local training. This verifies the pixel-space perturbation budget and the surrogate target-label success while keeping the V3 target out of poison generation.


In [ ]:
POISON_DEMO_RECIPE = str(attack_module_config.get("poison_demo_recipe", "pgd_default"))
POISON_DEMO_BATCH_SIZE = int(attack_module_config.get("poison_demo_batch_size", 8))
POISON_DEMO_VISUALIZE = bool(attack_module_config.get("visualize_poison_demo", True))


def _image_for_plot(image: torch.Tensor) -> np.ndarray:
    image_px = denormalize_pixels(image.unsqueeze(0)).squeeze(0)
    return image_px.detach().cpu().clamp(0.0, 1.0).permute(1, 2, 0).numpy()


def plot_poison_generation_samples(
    clean_inputs: torch.Tensor,
    poisoned_inputs: torch.Tensor,
    labels: torch.Tensor,
    poison_labels: torch.Tensor,
    *,
    max_images: int = 4,
) -> None:
    count = min(max_images, int(clean_inputs.size(0)))
    if count <= 0:
        return None
    fig, axes = plt.subplots(2, count, figsize=(2.6 * count, 4.8))
    if count == 1:
        axes = np.array(axes).reshape(2, 1)
    for idx in range(count):
        axes[0, idx].imshow(_image_for_plot(clean_inputs[idx]))
        axes[0, idx].set_title(f"clean y={int(labels[idx])}")
        axes[0, idx].axis("off")
        axes[1, idx].imshow(_image_for_plot(poisoned_inputs[idx]))
        axes[1, idx].set_title(f"poison y={int(poison_labels[idx])}")
        axes[1, idx].axis("off")
    plt.tight_layout()
    plt.show()


def demonstrate_poison_generation(
    recipe_name: str = "pgd_default",
    *,
    batch_size: int = 8,
    visualize: bool = True,
) -> dict:
    if recipe_name not in ATTACK_RECIPES:
        raise KeyError(f"Unknown poison demo recipe {recipe_name!r}. Available: {sorted(ATTACK_RECIPES)}")
    attack_cfg = _attach_attack_callable(ATTACK_RECIPES[recipe_name])
    client = make_malicious_client(attack_cfg, surrogate=surrogate_model, num_epochs=0)
    target_label = int(attack_cfg["attack"].get("target_label", ATTACK_BASE.get("target_label", 0)))
    inputs, labels = _next_eval_batch(batch_size, target_label=target_label)
    poisoned_inputs, poison_labels = craft_adversarial_batch(client, inputs, labels)
    metrics = _attack_metrics(surrogate_model, inputs, labels, poisoned_inputs, poison_labels)
    linf = pixel_linf_norm(inputs, poisoned_inputs, normalized=True)
    result = {
        "recipe": recipe_name,
        "attack_type": client.attack_type,
        "candidate_examples": int(inputs.size(0)),
        "poisoned_examples": int(poisoned_inputs.size(0)),
        "target_label": int(poison_labels[0].item()) if poison_labels.numel() else None,
        "epsilon": float(attack_cfg["attack"].get("epsilon", ATTACK_EPSILON)),
        "mean_pixel_linf": float(linf.mean().item()),
        "median_pixel_linf": float(linf.median().item()),
        "max_pixel_linf": float(linf.max().item()),
        "surrogate_clean_accuracy": float(metrics["clean_accuracy"]),
        "surrogate_robust_accuracy": float(metrics["robust_accuracy"]),
        "surrogate_target_label_success_rate": float(metrics["target_label_success_rate"]),
        "poison_generation_model": SURROGATE_MODEL_NAME,
        "poison_generation_checkpoint": str(surrogate_checkpoint_path),
        "poison_generation_gradient_source": "surrogate_model",
        "target_model": TARGET_MODEL_NAME,
        "target_gradients_used_for_poison_generation": False,
    }
    print(json.dumps(result, indent=2))
    if visualize:
        plot_poison_generation_samples(inputs, poisoned_inputs, labels, poison_labels)
    return {
        "metrics": result,
        "clean_inputs": inputs.detach().cpu(),
        "poisoned_inputs": poisoned_inputs.detach().cpu(),
        "clean_labels": labels.detach().cpu(),
        "poison_labels": poison_labels.detach().cpu(),
    }


poison_generation_demo = demonstrate_poison_generation(
    POISON_DEMO_RECIPE,
    batch_size=POISON_DEMO_BATCH_SIZE,
    visualize=POISON_DEMO_VISUALIZE,
)
poison_generation_demo["metrics"]


## 8. Random Noise, FGSM, and PGD on the Surrogate

Run the three attack recipes against MobileNetV2 only. This checks whether the loaded surrogate is vulnerable under the configured pixel budget.


In [ ]:
SURROGATE_ATTACK_RECIPES = ["random_noise", "fgsm_default", "pgd_default"]

if RUN_SURROGATE_ATTACKS:
    surrogate_attack_results = {
        recipe: evaluate_surrogate_attack(recipe)
        for recipe in SURROGATE_ATTACK_RECIPES
    }
else:
    surrogate_attack_results = {}
    print("Skipping surrogate attacks; set attack_module.run_surrogate_attacks=true in attack_module_config.yaml to run them.")

surrogate_attack_results


### Save and Plot Surrogate Attack Metrics

Save the surrogate-only attack summary and plot clean accuracy, robust accuracy, and target-label success.


In [ ]:
sur_attack_path = artifact_path("surrogate_attack_metrics", "module4_surrogate_attacks.json")
with sur_attack_path.open("w") as f:
    json.dump(surrogate_attack_results, f, indent=2)
print(f"Saved surrogate attack metrics to {sur_attack_path.resolve()}")


def plot_surrogate_attack_results(results: dict[str, dict]) -> None:
    if not results:
        print("No surrogate attack results to plot for this run mode.")
        return None
    df = pd.DataFrame(results).T
    labels = [row["attack"] for _, row in df.iterrows()]
    x = np.arange(len(df))
    width = 0.25
    plt.figure(figsize=(7, 4))
    plt.bar(x - width, df["clean_accuracy"], width=width, label="Clean acc")
    plt.bar(x, df["robust_accuracy"], width=width, label="Robust acc")
    plt.bar(x + width, df["surrogate_target_label_success_rate"], width=width, label="Target-label success")
    plt.xticks(x, labels)
    plt.ylabel("Percentage")
    plt.title("Surrogate attack sanity check")
    plt.legend()
    plt.tight_layout()
    out_path = artifact_path("surrogate_attack_plot", "surrogate_attack_success_by_attack.png")
    plt.savefig(out_path, dpi=150)
    print(f"Saved {out_path.resolve()}")
    return df[["attack", "epsilon", "clean_accuracy", "robust_accuracy", "surrogate_target_label_success_rate", "mean_linf", "max_linf"]]


plot_surrogate_attack_results(surrogate_attack_results)


## 9. Surrogate-to-Target Transfer

Craft adversarial examples with MobileNetV2 gradients, then evaluate those same tensors on the centralized MobileNetV3 target checkpoint.


In [ ]:
TRANSFER_ATTACK_RECIPES = ["random_noise", "fgsm_default", "pgd_default"]

if RUN_TRANSFER_ATTACKS:
    target_model = build_target_model_from_checkpoint()
    transfer_results = {
        recipe: evaluate_transfer_attack(recipe, target_model)
        for recipe in TRANSFER_ATTACK_RECIPES
    }
else:
    transfer_results = {}
    print("Skipping transfer attacks; set attack_module.run_transfer_attacks=true in attack_module_config.yaml to run them.")

transfer_path = artifact_path("transfer_metrics", "module4_transfer_results.json")
with transfer_path.open("w") as f:
    json.dump(transfer_results, f, indent=2)
print(f"Saved transfer metrics to {transfer_path.resolve()}")

if transfer_results:
    transfer_table = pd.DataFrame(transfer_results).T[
        [
            "attack",
            "epsilon",
            "target_clean_accuracy",
            "target_robust_accuracy",
            "target_model_target_label_success_rate",
            "surrogate_to_target_transfer_success_rate",
            "mean_linf",
            "max_linf",
        ]
    ]
    transfer_table
else:
    print("No transfer table for this run mode.")


## 10. Clean vs. Attacked FL

Run clean FL and PGD-poisoned FL from the same centralized V3 checkpoint using `attack_module.selected_algorithm`.

The registry supports FedAvg, FedAdam, FedAdagrad, FedYogi, and Scaffold. Malicious clients are algorithm-aware: FedOpt clients report `delta_y`, and SCAFFOLD clients preserve control-variate state while using the same surrogate poison path.


In [ ]:
FED_ATTACK_RECIPES = ["clean", "pgd_default"]

if RUN_FEDERATED_POISONING:
    federated_attack_results = sweep_attacks_on_server(
        algorithm_name=SELECTED_ALGORITHM,
        recipes=FED_ATTACK_RECIPES,
    )
else:
    federated_attack_results = {}
    print("Skipping federated poisoning; set attack_module.run_federated_poisoning=true in attack_module_config.yaml to run it.")

federated_attack_results


### Save FL Metrics

Save clean baseline metrics separately and save the clean-vs-attacked comparison in `module4_federated_attacks.json`.


In [ ]:
if federated_attack_results.get("clean"):
    baseline_results[SELECTED_ALGORITHM] = federated_attack_results["clean"]
    baseline_path = artifact_path("baseline_metrics", "module4_federated_baseline.json")
    with baseline_path.open("w") as f:
        json.dump(baseline_results, f, indent=2)
    print(f"Saved clean {SELECTED_ALGORITHM} baseline metrics to {baseline_path.resolve()}")

fed_attack_path = artifact_path("federated_attack_metrics", "module4_federated_attacks.json")
with fed_attack_path.open("w") as f:
    json.dump(federated_attack_results, f, indent=2)
print(f"Saved federated attack metrics to {fed_attack_path.resolve()}")


### Federated Attack Sanity Check

Compare final clean and attacked FL accuracy for the selected algorithm. `global_target_label_asr` is measured on the final MobileNetV3 global model, not on the surrogate.


In [ ]:
clean_acc = federated_attack_results.get("clean", {}).get("final_accuracy")
attack_summary = federated_attack_results.get("pgd_default", {})
attack_acc = attack_summary.get("final_accuracy")
global_asr = attack_summary.get("final_global_target_label_asr", attack_summary.get("global_target_label_asr"))
if clean_acc is not None and attack_acc is not None:
    delta = clean_acc - attack_acc
    message = f"{SELECTED_ALGORITHM} clean accuracy: {clean_acc:.2f}%  |  attacked: {attack_acc:.2f}%  |  drop: {delta:.2f} pts"
    if global_asr is not None:
        message += f"  |  global target-label ASR: {global_asr:.2f}%"
    print(message)
else:
    print(f"Run the clean and attacked {SELECTED_ALGORITHM} cell before executing this check.")


### Federated Attack Impact Plot

Plot the clean and attacked accuracy curves across rounds for the selected algorithm. Inspect whether the attacked curve diverges after the configured attack start round.


In [ ]:
def plot_federated_attack_results(results: dict[str, dict]) -> None:
    clean = results.get("clean", {})
    attacked = results.get("pgd_default", {})
    if not clean or not attacked:
        print(f"Run the clean and attacked {SELECTED_ALGORITHM} cells before plotting.")
        return

    clean_history = clean.get("history", {})
    attacked_history = attacked.get("history", {})
    clean_acc = clean_history.get("accuracy", [])
    attack_acc = attacked_history.get("accuracy", [])

    plt.figure(figsize=(7, 4))
    if clean_acc:
        plt.plot(range(1, len(clean_acc) + 1), clean_acc, marker="o", label=f"Clean {SELECTED_ALGORITHM}")
    if attack_acc:
        plt.plot(range(1, len(attack_acc) + 1), attack_acc, marker="o", label=f"Attacked {SELECTED_ALGORITHM}")
    start_round = int(attack_defaults.get("start_round", 0))
    if start_round:
        plt.axvline(start_round, linestyle="--", color="black", alpha=0.6, label="Attack starts")
    plt.xlabel("Round")
    plt.ylabel("Accuracy (%)")
    plt.title(f"Clean vs attacked {SELECTED_ALGORITHM} from V3 checkpoint")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out_path = artifact_path("federated_attack_plot", "attack_accuracy.png")
    plt.savefig(out_path, dpi=150)
    print(f"Saved {out_path.resolve()}")

    clean_asr = clean_history.get("global_target_label_asr", [])
    attacked_asr = attacked_history.get("global_target_label_asr", [])
    if not clean_asr and not attacked_asr:
        print("No per-round global_target_label_asr history available to plot.")
        return

    plt.figure(figsize=(7, 4))
    if clean_asr:
        plt.plot(range(1, len(clean_asr) + 1), clean_asr, marker="o", label=f"Clean {SELECTED_ALGORITHM}")
    if attacked_asr:
        plt.plot(range(1, len(attacked_asr) + 1), attacked_asr, marker="o", label=f"Attacked {SELECTED_ALGORITHM}")
    if start_round:
        plt.axvline(start_round, linestyle="--", color="black", alpha=0.6, label="Attack starts")
    plt.xlabel("Round")
    plt.ylabel("Global target-label ASR (%)")
    plt.title(f"Global target-label ASR for {SELECTED_ALGORITHM}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    asr_path = artifact_path("federated_attack_asr_plot", "global_target_label_asr.png")
    plt.savefig(asr_path, dpi=150)
    print(f"Saved {asr_path.resolve()}")


plot_federated_attack_results(federated_attack_results)


## 11. Attack Recipe Sweep

Optionally run clean FL once and then rerun attacked FL for each configured poisoning recipe on the selected algorithm. This compares random noise, FGSM, and PGD as federated data-poisoning recipes using the same V3 initialization, V2 surrogate, client split settings, and malicious-client configuration.


In [ ]:
attack_recipe_sweep_payload = {}
attack_recipe_sweep_results = {}
attack_recipe_sweep_rows = []

if not RUN_ATTACK_RECIPE_SWEEP:
    print("Skipping attack-recipe sweep; set attack_module.run_attack_recipe_sweep=true in attack_module_config.yaml to run it.")
else:
    attack_recipe_sweep_payload = run_attack_recipe_sweep(
        algorithm_name=SELECTED_ALGORITHM,
        recipes=ATTACK_RECIPE_SWEEP_RECIPES,
    )
    attack_recipe_sweep_results = attack_recipe_sweep_payload["results"]
    attack_recipe_sweep_rows = attack_recipe_sweep_payload["summary_table"]

attack_recipe_sweep_columns = [
    "algorithm",
    "attack_recipe",
    "final_clean_accuracy",
    "final_attacked_accuracy",
    "accuracy_drop",
    "surrogate_poison_success_rate",
    "global_target_label_asr",
    "poisoned_examples",
    "malicious_fraction",
    "num_rounds",
]
attack_recipe_sweep_table = pd.DataFrame(attack_recipe_sweep_rows)
if not attack_recipe_sweep_table.empty:
    attack_recipe_sweep_table = attack_recipe_sweep_table[attack_recipe_sweep_columns]
attack_recipe_sweep_table


### Save and Plot Attack-Recipe Sweep

When enabled, save the full recipe-sweep payload and plot final clean accuracy, final attacked accuracy, surrogate poison success, and `global_target_label_asr` by poisoning recipe.


In [ ]:
attack_recipe_sweep_path = artifact_path("attack_recipe_sweep_metrics", "module4_attack_recipe_sweep.json")
if attack_recipe_sweep_rows:
    with attack_recipe_sweep_path.open("w") as f:
        json.dump(attack_recipe_sweep_payload, f, indent=2)
    print(f"Saved attack-recipe sweep results to {attack_recipe_sweep_path.resolve()}")
else:
    print("No attack-recipe sweep rows to save for this run mode.")


def plot_attack_recipe_sweep(rows: list[dict] | None) -> None:
    if not rows:
        print("No attack-recipe sweep results available to plot.")
        return

    labels = [attack_label(row["attack_recipe"]) for row in rows]
    x = np.arange(len(labels))
    width = 0.35

    def _plot_value(value):
        return np.nan if value is None else float(value)

    clean_acc = [_plot_value(row.get("final_clean_accuracy")) for row in rows]
    attacked_acc = [_plot_value(row.get("final_attacked_accuracy")) for row in rows]
    global_asr = [_plot_value(row.get("global_target_label_asr")) for row in rows]
    poison_success = [_plot_value(row.get("surrogate_poison_success_rate")) for row in rows]

    plt.figure(figsize=(8, 4))
    plt.bar(x - width / 2, clean_acc, width=width, label="Final clean accuracy")
    plt.bar(x + width / 2, attacked_acc, width=width, label="Final attacked accuracy")
    plt.plot(x, global_asr, marker="s", linestyle="--", color="tab:red", label="Global target-label ASR")
    plt.plot(x, poison_success, marker="^", linestyle=":", color="tab:green", label="Surrogate poison success")
    plt.xticks(x, labels)
    plt.ylabel("Percentage")
    plt.title(f"Attack-recipe sweep for {SELECTED_ALGORITHM}")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out_path = artifact_path("attack_recipe_sweep_plot", "attack_recipe_sweep.png")
    plt.savefig(out_path, dpi=150)
    print(f"Saved {out_path.resolve()}")


plot_attack_recipe_sweep(attack_recipe_sweep_rows)


## 12. Malicious Fraction Sweep

Optionally vary the fraction of malicious clients and rerun attacked FL for the selected algorithm. The clean baseline is reused because it does not depend on malicious fraction.


In [ ]:
malicious_grid = MALICIOUS_FRACTION_GRID
fraction_sweep_results = {}
fraction_sweep_rows = []
if not RUN_MALICIOUS_FRACTION_SWEEP:
    print("Skipping malicious-fraction sweep; set attack_module.run_malicious_fraction_sweep=true in attack_module_config.yaml to run it.")
else:
    if SELECTED_ALGORITHM not in baseline_results:
        baseline_results[SELECTED_ALGORITHM] = run_attack_recipe_on_server(
            "clean",
            algorithm_name=SELECTED_ALGORITHM,
            malicious_fraction=0.0,
        )
    for frac in malicious_grid:
        fraction_sweep_results[frac] = sweep_attacks_on_server(
            algorithm_name=SELECTED_ALGORITHM,
            recipes=["clean", "pgd_default"],
            malicious_fraction=frac,
        )

    for frac, results in fraction_sweep_results.items():
        fraction_sweep_rows.append(
            build_clean_attacked_summary_row(
                SELECTED_ALGORITHM,
                results,
                attack_recipe="pgd_default",
                extra={"malicious_fraction": frac},
            )
        )

fraction_sweep_rows


### Save and Plot the Malicious-Fraction Sweep

When the optional sweep is enabled, save the sweep table and plot final attacked accuracy, global target-label ASR, and surrogate poison success.


In [ ]:
fraction_path = artifact_path("fraction_sweep_metrics", "module4_fraction_sweep.json")
if fraction_sweep_rows:
    with fraction_path.open("w") as f:
        json.dump(fraction_sweep_rows, f, indent=2)
    print(f"Saved sweep results to {fraction_path.resolve()}")
else:
    print("No malicious-fraction sweep rows to save for this run mode.")


def plot_fraction_sweep(rows: list[dict] | None) -> None:
    if not rows:
        print("No fraction sweep results available to plot.")
        return
    fractions = [row["malicious_fraction"] for row in rows]
    accuracies = [row.get("final_attacked_accuracy") for row in rows]
    global_target_label_asr = [row.get("global_target_label_asr", 0.0) for row in rows]
    surrogate_poison_success = [row.get("surrogate_poison_success_rate", 0.0) for row in rows]

    plt.figure(figsize=(7, 4))
    plt.plot(fractions, accuracies, marker="o", label="Final attacked accuracy")
    plt.plot(fractions, global_target_label_asr, marker="s", label="Global target-label ASR")
    plt.plot(fractions, surrogate_poison_success, marker="^", label="Surrogate poison success")
    plt.xlabel("Malicious fraction")
    plt.ylabel("Percentage")
    plt.title(f"Impact of malicious fraction on {SELECTED_ALGORITHM}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out_path = artifact_path("fraction_sweep_plot", "malicious_fraction_sweep.png")
    plt.savefig(out_path, dpi=150)
    print(f"Saved {out_path.resolve()}")


plot_fraction_sweep(fraction_sweep_rows)


## 13. Algorithm Comparison

Optionally run clean and attacked FL for multiple configured algorithms under the same attack recipe. This keeps the data settings, target checkpoint, surrogate checkpoint, malicious-client selection seed, and attack recipe fixed where possible so the summary isolates algorithm behavior.


In [ ]:
algorithm_comparison_payload = {}
algorithm_comparison_results = {}
algorithm_comparison_rows = []

if not RUN_ALGORITHM_COMPARISON:
    print("Skipping algorithm comparison; set attack_module.run_algorithm_comparison=true in attack_module_config.yaml to run it.")
else:
    algorithm_comparison_payload = run_algorithm_comparison(
        algorithms=ALGORITHM_COMPARISON_ALGORITHMS,
        attack_recipe=ALGORITHM_COMPARISON_ATTACK_RECIPE,
    )
    algorithm_comparison_results = algorithm_comparison_payload["results"]
    algorithm_comparison_rows = algorithm_comparison_payload["summary_table"]

algorithm_comparison_columns = [
    "algorithm",
    "attack_recipe",
    "final_clean_accuracy",
    "final_attacked_accuracy",
    "accuracy_drop",
    "surrogate_poison_success_rate",
    "global_target_label_asr",
    "poisoned_examples",
    "malicious_fraction",
    "num_rounds",
]
algorithm_comparison_table = pd.DataFrame(algorithm_comparison_rows)
if not algorithm_comparison_table.empty:
    algorithm_comparison_table = algorithm_comparison_table[algorithm_comparison_columns]
algorithm_comparison_table


### Save and Plot Algorithm Comparison

When enabled, save the full clean-vs-attacked results plus a compact summary table. The plot compares final clean accuracy, final attacked accuracy, surrogate poison success, and `global_target_label_asr` by algorithm.


In [ ]:
algorithm_comparison_path = artifact_path("algorithm_comparison_metrics", "module4_algorithm_comparison.json")
if algorithm_comparison_rows:
    with algorithm_comparison_path.open("w") as f:
        json.dump(algorithm_comparison_payload, f, indent=2)
    print(f"Saved algorithm comparison results to {algorithm_comparison_path.resolve()}")

    baseline_path = artifact_path("baseline_metrics", "module4_federated_baseline.json")
    with baseline_path.open("w") as f:
        json.dump(baseline_results, f, indent=2)
    print(f"Updated clean baseline metrics for compared algorithms at {baseline_path.resolve()}")
else:
    print("No algorithm comparison rows to save for this run mode.")


def plot_algorithm_comparison(rows: list[dict] | None) -> None:
    if not rows:
        print("No algorithm comparison results available to plot.")
        return

    labels = [row["algorithm"] for row in rows]
    x = np.arange(len(labels))
    width = 0.35

    def _plot_value(value):
        return np.nan if value is None else float(value)

    clean_acc = [_plot_value(row.get("final_clean_accuracy")) for row in rows]
    attacked_acc = [_plot_value(row.get("final_attacked_accuracy")) for row in rows]
    global_asr = [_plot_value(row.get("global_target_label_asr")) for row in rows]
    poison_success = [_plot_value(row.get("surrogate_poison_success_rate")) for row in rows]

    plt.figure(figsize=(8, 4))
    plt.bar(x - width / 2, clean_acc, width=width, label="Final clean accuracy")
    plt.bar(x + width / 2, attacked_acc, width=width, label="Final attacked accuracy")
    plt.plot(x, global_asr, marker="s", linestyle="--", color="tab:red", label="Global target-label ASR")
    plt.plot(x, poison_success, marker="^", linestyle=":", color="tab:green", label="Surrogate poison success")
    plt.xticks(x, labels, rotation=20, ha="right")
    plt.ylabel("Percentage")
    plt.title(f"Algorithm comparison under {attack_label(ALGORITHM_COMPARISON_ATTACK_RECIPE)} poisoning")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out_path = artifact_path("algorithm_comparison_plot", "algorithm_comparison.png")
    plt.savefig(out_path, dpi=150)
    print(f"Saved {out_path.resolve()}")


plot_algorithm_comparison(algorithm_comparison_rows)


## 14. Fast Synthetic Validation

The standalone smoke path validates all supported FL algorithms on tiny synthetic data without Imagenette or trained checkpoints. It is optional and intended for wiring checks, not interpretation.


In [ ]:
RUN_FAST_VALIDATION = bool(attack_module_config.get("run_fast_validation", False))
if RUN_FAST_VALIDATION:
    from smoke_validation import run_fast_validation
    fast_validation_path = artifact_path("fast_validation_metrics", "module4_fast_validation.json")
    fast_validation_results = run_fast_validation(artifact_path=fast_validation_path)
    print(f"Saved fast validation results to {fast_validation_path.resolve()}")
else:
    fast_validation_results = []
    print("Skipping fast synthetic validation; set attack_module.run_fast_validation=true to run it.")

fast_validation_results


## Final Interpretation and Transition to Module 5

Use the generated artifacts to write a short interpretation of the attack pipeline. A complete answer should distinguish corruption robustness, gradient-based adversarial examples, black-box transfer, and malicious-client poisoning.

Checkpoint questions:

1. Did random noise hurt the surrogate?
2. Did FGSM/PGD hurt more than random noise?
3. Did surrogate-crafted attacks transfer to the centralized MobileNetV3 target?
4. Starting from the same V3 checkpoint, did malicious clients reduce global accuracy for the selected FL algorithm?
5. When attack-recipe sweep is enabled, which recipe caused the largest attacked-accuracy drop?
6. When algorithm comparison is enabled, which algorithm had the smallest attacked-accuracy drop under the same recipe?
7. Did surrogate poison success increase with malicious fraction?
8. Did final global-model target-label ASR move with malicious fraction, attack recipe, or algorithm choice?
9. Why is surrogate poison success not the same thing as final global-model target-label ASR?
10. Why does this motivate robust aggregation?

**Transition to Module 5:** Plain averaging can include malicious and honest updates together, so even a small malicious fraction can influence the global model. Module 5 asks whether robust aggregation, clipping, median, trimmed mean, Krum, Multi-Krum, and related defenses can preserve clean accuracy while reducing attacked accuracy loss and global target-label ASR.


## Artifact Guide

After a default run, inspect these files in `artifacts/` to answer the main Module 4 questions. The attack-recipe sweep, malicious-fraction sweep, and algorithm-comparison artifacts are written only when the corresponding optional modes are enabled.

| Artifact | What it answers |
| --- | --- |
| `module4_attack_config_used.json` | Which config, device, attack budget, target label, selected algorithm, malicious client ids, and FL settings produced this run? |
| `module4_target_training.json` | How well did the centralized MobileNetV3 target train before attacks? |
| `module4_v3_target.pt` | Which V3 checkpoint initialized transfer evaluation and clean-vs-attacked FL? |
| `module4_surrogate.json` | Did the centralized MobileNetV2 surrogate train well enough to be a credible attacker model? |
| `module4_surrogate.pt` | Which surrogate checkpoint crafted the FGSM and PGD examples? |
| Poison generation dry run output | Before FL, did the configured malicious-client attack path create bounded V2-surrogate poisons without target gradients? |
| `module4_surrogate_attacks.json` | On the surrogate, how do random noise, FGSM, and PGD compare for clean accuracy, robust accuracy, and target-label success? |
| `surrogate_attack_success_by_attack.png` | Which attack most clearly separates adversarial behavior from random corruption on the surrogate? |
| `module4_transfer_results.json` | Did MobileNetV2-crafted examples transfer to the MobileNetV3 target? |
| `module4_federated_baseline.json` | How did the selected algorithm behave without poisoning when initialized from the V3 checkpoint? |
| `module4_federated_attacks.json` | Under the default malicious fraction, how did attacked FL change global accuracy, poisoned-example counts, surrogate poison success, and final `global_target_label_asr`? |
| `attack_accuracy.png` | When the attack started, did attacked FL diverge from clean FL? |
| `global_target_label_asr.png` | Did held-out non-target examples increasingly map to the configured target label over rounds? |
| `module4_attack_recipe_sweep.json` | Optional: for the selected algorithm, how do random noise, FGSM, and PGD poisoning compare on final clean accuracy, final attacked accuracy, accuracy drop, surrogate poison success, and `global_target_label_asr`? |
| `attack_recipe_sweep.png` | Optional: which poisoning recipe most changes attacked accuracy or target-label behavior for the selected algorithm? |
| `module4_fraction_sweep.json` | Optional: as malicious-client fraction changes, how do final attacked accuracy, `global_target_label_asr`, poisoned examples, and surrogate poison success change? |
| `malicious_fraction_sweep.png` | Optional: what visual tradeoff appears between attacked accuracy, global target-label ASR, and surrogate poison success as more clients become malicious? |
| `module4_algorithm_comparison.json` | Optional: under one attack recipe, how do configured algorithms compare on final clean accuracy, final attacked accuracy, accuracy drop, surrogate poison success, and `global_target_label_asr`? |
| `algorithm_comparison.png` | Optional: which algorithm best preserves attacked accuracy while limiting global target-label ASR under the shared recipe? |
